# 批次对照：一次启动，现场逐项按键，最后统一交结果
本批最多8项：−30、−24、−18、−12 dB，每档先旁路再抵消，每项2秒。沿用标定模型，Codec音量40。无需重复零模型验收。
保持箱体、接线、手机位置和播放音量不变，手机循环播放普通音量人声。不戴耳机，能随手切断音响电源。出现尖叫立即断电，不要等2秒结束。
## 第1格：文件校验和载入（不会播放）
先执行下面代码格。必须看到 BATCH_PACKAGE_OK。哈希报错时停止，检查本次四个更新文件是否上传齐全。


In [ ]:
import hashlib, json, importlib
from pathlib import Path
manifest = json.loads(Path('manifest.json').read_text())
for name in ('audio_closed.bit', 'audio_closed.hwh', 'closed_trial.py', 'closed_tools.py', 'residual_tools.py', 'closed_batch.py'):
    assert hashlib.sha256(Path(name).read_bytes()).hexdigest() == manifest['hashes']['board/audio_closed/'+name], name
import closed_trial as trial_module
import closed_batch as batch_module
importlib.reload(trial_module)
importlib.reload(batch_module)
print('BATCH_PACKAGE_OK')


## 第2格：运行整批
执行下面格一次。出现输入框时按提示输入，再按Enter（不是Shift+Enter）。
1. 输入 READY：确认现场就绪且同意板卡本地保存。否则输入q退出。
2. 程序先配置硬件；看到“配置已完成，当前静音”后，你准备听时输入 START。此时才开始该项2秒试验。
3. 等结束及恢复完成，输入听感：0=没听见且结束安静；1=听见人声且结束安静。两者允许准备下一项。2=杂音；3=尖叫/持续增长；4=结束仍响；q=停止。这几项停止整批，不再提高增益。
4. 重复START和听感选择，最多8项，不需要逐次返回聊天。可以随时在START提示处输入q停止。
不要重复运行此格来跳过失败项。结束后左侧[*]恢复数字，输出会给出summary.json位置。


In [ ]:
batch_result = batch_module.run_batch(runner=trial_module.closed_trial)


## 结束后只发一次汇总
在Jupyter文件列表打开 batch_results → 本次 batch_… 文件夹。下载 summary.json 发回；summary.csv用于表格查看。原始NPZ不用上传。
即使中途停止，也返回该目录summary.json。错误/饱和会自动停止后续项目，但听到尖叫时仍要现场断电。
没有全程频谱监测，只有末尾约0.336秒PCM及全程饱和计数。不会自动算MSG或ASG，也不会自动停止尚未饱和的啸叫。本批只找可用增益范围和A/B异常，为后续阈值测试提供依据。
